# ДЗ2 NLP: быстрый mixed inference на одной T4

Private, Internet off. Входы: `thisarivson/nlp-hw02-l2-live` (FP16 l2-28800 обеих C и журналы), `thisarivson/nlp-hw02-26b-inputs` (tokenizer/test/heldout/reference). Обучения нет.

Один запуск: два холодных test1919 в новых процессах, затем тот же K4+K4+greedy C-main на tune2048. `answers.jsonl` — первый test; каждая строка содержит только `{"code": "..."}`. Gate: все test ≤120 с, ни одного fallback, tune отличается от 1492/2048 не более чем на 1 п.п. 180 с — обязательный потолок; ≤120 с — цель v3.

Таймер каждого test начинается при старте supervisor-процесса до импорта Torch и чтения test, заканчивается после закрытия answers. Greedy b1024; sampling b256×K4 =1024 lanes, length buckets и OOM-halving. CPU wall3/CPU2/768MiB, все ядра, дедупликация кода без удаления голосов; исполнение перекрывает следующие GPU-батчи. SDPA4D physical cross-KV, FP16, cap512, T0.7, seed270042 отдельно на модель, untempered mean LP с EOS. Numeric isclose1e-6 graph → count/greedy/mean LP/order. Код: greedy в победившей компоненте, иначе максимальная mean LP.

K4/b256 с length bucketing изменяет RNG относительно первых4 из L2 K8/b32. Старый результат воспроизводится по журналам; новые K4 оцениваются с допуском ±1 п.п. Автоматических дополнительных GPU-прогонов нет. Два test должны дать побайтно одинаковые ответы; выбранный код повторно исполняется в свежем сервисе внутри таймера. Сессия ≤15 мин, watchdog 840/870 с. Никаких secrets, pip, загрузок или публикаций. Требуется исходный runtime L2: Torch2.11.0+cu128/CUDA12.8.


In [ ]:
import time,os,sys,json
from pathlib import Path
SESSION_START=time.monotonic()
os.environ['CUDA_VISIBLE_DEVICES']='0'
os.environ['TOKENIZERS_PARALLELISM']='false'
OUTPUT=Path('/kaggle/working/final-inference') if Path('/kaggle/input').exists() else Path('final-inference')
OUTPUT.mkdir(parents=True,exist_ok=True)
assert not (OUTPUT/'answers.jsonl').exists(),'Use a new OUTPUT directory for another session; saved answers are retained'
MODULES=OUTPUT/'src';MODULES.mkdir(exist_ok=True)
SOURCES={'final_core.py': "# Generated by build_final_inference.py from exp26_core.py; do not edit.\nimport math,time\nimport torch\nfrom torch import nn\nfrom torch.nn import functional as F\nPAD,BOS,EOS,TOKEN_OFFSET=0,1,2,3\nVOCAB=8192\nMAX_POSITIONS=2048\nSEED=42\n\nclass SwiGLUInput(nn.Module):\n def __init__(self,width,hidden):\n  super().__init__();self.gate=nn.Linear(width,hidden);self.value=nn.Linear(width,hidden)\n def forward(self,x):return F.silu(self.gate(x))*self.value(x)\n\n\ndef replace_ffn_with_swiglu(transformer,width,hidden):\n with torch.random.fork_rng(devices=[]):\n  torch.set_rng_state(torch.Generator().manual_seed(SEED).get_state())\n  for layer in list(transformer.encoder.layers)+list(transformer.decoder.layers):\n   layer.linear1=SwiGLUInput(width,hidden);layer.linear2=nn.Linear(hidden,width);layer.activation=nn.Identity()\n   if isinstance(layer,nn.TransformerEncoderLayer):layer.activation_relu_or_gelu=0\n   for module in [layer.linear1.gate,layer.linear1.value,layer.linear2]:nn.init.xavier_uniform_(module.weight)\n\n\nclass SimpleTransformer(nn.Module):\n def __init__(self,d_model=256,heads=4,encoder_layers=4,decoder_layers=4,swiglu_dim=680,\n              vocab=VOCAB,max_positions=MAX_POSITIONS,dropout=.1):\n  super().__init__();self.width=d_model;self.vocab=vocab\n  self.tokens=nn.Embedding(vocab,d_model,padding_idx=PAD);self.positions=nn.Embedding(max_positions,d_model)\n  self.transformer=nn.Transformer(d_model=d_model,nhead=heads,num_encoder_layers=encoder_layers,\n   num_decoder_layers=decoder_layers,dim_feedforward=4*d_model,dropout=dropout,batch_first=True,norm_first=True)\n  self.transformer.encoder.enable_nested_tensor=False;self.transformer.encoder.use_nested_tensor=False\n  self.head=nn.Linear(d_model,vocab);replace_ffn_with_swiglu(self.transformer,d_model,swiglu_dim)\n def embed(self,ids):\n  if ids.shape[1]>self.positions.num_embeddings:raise ValueError('Position overflow')\n  return self.tokens(ids)*math.sqrt(self.width)+self.positions(torch.arange(ids.shape[1],device=ids.device))\n def decode_prefix(self,tgt,memory,source_padding):\n  mask=torch.ones(tgt.shape[1],tgt.shape[1],device=tgt.device,dtype=torch.bool).triu(1)\n  return self.transformer.decoder(self.embed(tgt),memory,tgt_mask=mask,tgt_key_padding_mask=tgt.eq(PAD),memory_key_padding_mask=source_padding)\n def forward(self,src,tgt):\n  memory=self.transformer.encoder(self.embed(src),src_key_padding_mask=src.eq(PAD))\n  return self.head(self.decode_prefix(tgt,memory,src.eq(PAD)))\n @torch.inference_mode()\n def generate(self,src,max_new_tokens=512):\n  self.eval();memory=self.transformer.encoder(self.embed(src),src_key_padding_mask=src.eq(PAD))\n  tgt=torch.full((len(src),1),BOS,device=src.device,dtype=torch.long);done=torch.zeros(len(src),device=src.device,dtype=torch.bool)\n  for _ in range(max_new_tokens):\n   logits=self.head(self.decode_prefix(tgt,memory,src.eq(PAD))[:,-1]);logits[:,[PAD,BOS]]=-torch.inf\n   token=logits.argmax(-1).masked_fill(done,PAD);done|=token.eq(EOS);tgt=torch.cat([tgt,token[:,None]],dim=1)\n   if bool(done.all()):break\n  return tgt[:,1:]\n\n\ndef encode(tokenizer,text):return [i+TOKEN_OFFSET for i in tokenizer.encode(text,add_special_tokens=False).ids]\n\n\ndef decode_bytes(tokenbytes,ids):\n chunks=[]\n for i in ids:\n  if i==EOS:break\n  if i>=TOKEN_OFFSET:chunks.append(tokenbytes[i])\n return b''.join(chunks)\n\n\ndef active_tokens(ids):\n ids=[int(i)for i in ids if i!=PAD]\n return ids[:ids.index(EOS)+1]if EOS in ids else ids\n\n\nclass BudgetExpired(Exception):pass\n\n\ndef check_deadline(deadline):\n if deadline is not None and time.monotonic()>=deadline:raise BudgetExpired('Session budget reached; completed run boundaries are preserved')\n\n\ndef sync(device):\n if device.type=='cuda':torch.cuda.synchronize(device)\n\n", 'final_fast.py': '# Generated by build_final_inference.py from exp27b_fast.py; do not edit.\nimport math\nimport torch\nfrom torch.nn import functional as F\nimport final_core as C\n\nclass Context:\n    @torch.inference_mode()\n    def __init__(self,model,src,capacity):\n        self.model,self.src,self.capacity=model,src,capacity\n        self.width=model.width;self.heads=model.transformer.decoder.layers[0].self_attn.num_heads;self.dim=self.width//self.heads\n        memory=model.transformer.encoder(model.embed(src),src_key_padding_mask=src.eq(C.PAD))\n        self.allowed=src.ne(C.PAD)[:,None,None,:];self.positions=model.positions(torch.arange(capacity,device=src.device));self.cross=[]\n        for layer in model.transformer.decoder.layers:\n            a=layer.multihead_attn;k,v=F.linear(memory,a.in_proj_weight[self.width:],a.in_proj_bias[self.width:]).chunk(2,-1)\n            self.cross.append(tuple(t.reshape(len(src),-1,self.heads,self.dim).transpose(1,2).contiguous() for t in (k,v)))\n\n\nclass Decoder:\n    @torch.inference_mode()\n    def __init__(self,context,k):\n        self.c,self.k,self.t=context,k,0;self.n=len(context.src)*k\n        self.allowed=context.allowed.repeat_interleave(k,dim=0)\n        self.cross=[tuple(t.repeat_interleave(k,dim=0).contiguous() for t in pair) for pair in context.cross]\n        self.cross_bytes=sum(t.numel()*t.element_size() for pair in self.cross for t in pair)\n        shape=(self.n,context.heads,context.capacity,context.dim)\n        self.self_kv=[(context.positions.new_empty(shape),context.positions.new_empty(shape)) for _ in context.cross]\n        self.valid=torch.zeros(self.n,context.capacity,dtype=torch.bool,device=context.src.device)\n    def heads(self,x):return x.reshape(self.n,1,self.c.heads,self.c.dim).transpose(1,2)\n    def attend(self,a,q,k,v,mask):\n        assert q.ndim==k.ndim==v.ndim==4\n        y=F.scaled_dot_product_attention(q,k,v,attn_mask=mask,dropout_p=0.,is_causal=False)\n        return a.out_proj(y.transpose(1,2).contiguous().reshape(self.n,1,self.c.width))\n    @torch.inference_mode()\n    def step(self,tokens):\n        c=self.c;t=self.t;x=c.model.tokens(tokens[:,None])*math.sqrt(c.width)+c.positions[t][None,None,:]\n        self.valid[:,t]=tokens.ne(C.PAD)\n        for layer,(sk,sv),(ck,cv) in zip(c.model.transformer.decoder.layers,self.self_kv,self.cross):\n            a=layer.self_attn;q,k,v=F.linear(layer.norm1(x),a.in_proj_weight,a.in_proj_bias).chunk(3,-1)\n            sk[:,:,t:t+1]=self.heads(k);sv[:,:,t:t+1]=self.heads(v)\n            x=x+layer.dropout1(self.attend(a,self.heads(q),sk[:,:,:t+1],sv[:,:,:t+1],self.valid[:,None,None,:t+1]))\n            a=layer.multihead_attn;q=F.linear(layer.norm2(x),a.in_proj_weight[:c.width],a.in_proj_bias[:c.width])\n            x=x+layer.dropout2(self.attend(a,self.heads(q),ck,cv,self.allowed))\n            x=x+layer.dropout3(layer.linear2(layer.dropout(layer.activation(layer.linear1(layer.norm3(x))))))\n        if c.model.transformer.decoder.norm is not None:x=c.model.transformer.decoder.norm(x)\n        self.t+=1;return c.model.head(x[:,0])\n\n\nfrom final_decode import decode\n', 'final_choose.py': "# Generated by build_final_inference.py from exp27_decode.py; do not edit.\nimport math\n\ndef choose(row,cfg):\n    greedy=row['greedy']\n    if cfg['mode']=='greedy': return greedy\n    samples=row['samples'][str(cfg['temperature'])][:cfg['k']]\n    if cfg['mode']=='fallback':\n        return greedy if greedy['execution']['status']=='numeric' else next((s for s in samples if s['execution']['status']=='numeric'),greedy)\n    votes=([greedy] if cfg['greedy_vote'] else [])+samples\n    good=[v for v in votes if v['execution']['status']=='numeric']\n    if not good:return greedy\n    # isclose is not transitive: explicitly use connected components of its pairwise graph.\n    parent=list(range(len(good)))\n    def root(i):\n        while parent[i]!=i:parent[i]=parent[parent[i]];i=parent[i]\n        return i\n    for i in range(len(good)):\n        for j in range(i):\n            if math.isclose(good[i]['execution']['value'],good[j]['execution']['value'],rel_tol=1e-6,abs_tol=1e-6):\n                parent[root(i)]=root(j)\n    clusters={}\n    for i,v in enumerate(good):clusters.setdefault(root(i),[]).append(v)\n    def key(vs):\n        contains=greedy['execution']['status']=='numeric' and any(math.isclose(v['execution']['value'],greedy['execution']['value'],rel_tol=1e-6,abs_tol=1e-6) for v in vs)\n        return len(vs),contains,sum(v['mean_logprob'] for v in vs)/len(vs)\n    winning=max(clusters.values(),key=key)\n    return max(winning,key=lambda v:v['mean_logprob'])\n\n", 'final_split.py': "# Generated by build_final_inference.py from l2_sampler.py; do not edit.\nimport hashlib\n\ndef fixed_split(n=4096):\n    ordered = sorted(range(n), key=lambda i: hashlib.sha256(\n        f'candidate-selection-v1:heldout4k:{i}'.encode()).digest())\n    return {'tune2048': sorted(ordered[:n//2]), 'eval2048': sorted(ordered[n//2:])}\n\n", 'final_representation.py': "# Generated by build_final_inference.py from s3_representation.py; do not edit.\nfrom tokenizers import Tokenizer,AddedToken\nMODES={'header_tail':'<|header_tail|>','header':'<|header|>','raw':'<|raw|>'}\nNUMBERS=[f'<|N{i}|>' for i in range(1,21)]\n\ndef restore_compact(text):\n    mode = next((k for k, token in MODES.items() if text.startswith(token)), None)\n    if mode is None:\n        raise ValueError('Missing representation mode')\n    fragment = text[len(MODES[mode]):]\n    if mode == 'raw':\n        return fragment\n    if mode == 'header':\n        return 'def solve() -> int:\\n' + fragment\n    lines = fragment.splitlines(keepends=True)\n    if not lines or not lines[-1].startswith('    '):\n        raise ValueError('Missing final result expression')\n    lines[-1] = '    result = ' + lines[-1][4:]\n    body = ''.join(lines)\n    return 'def solve() -> int:\\n' + body + ('' if body.endswith('\\n') else '\\n') + '    return result\\n'\n\n\ndef tokenizer(path, variant):\n    tok = Tokenizer.from_file(str(path))\n    assert tok.get_vocab_size() == 8189\n    added = list(MODES.values()) if variant == 30 else NUMBERS\n    assert tok.add_special_tokens([AddedToken(t, normalized=False, special=True) for t in added]) == len(added)\n    # Original byte alphabet is unchanged; added specials decode literally, never get skipped.\n    visible = list(range(33,127)) + list(range(161,173)) + list(range(174,256))\n    bchars = {b:chr(b) for b in visible}\n    for i,b in enumerate(b for b in range(256) if b not in bchars):\n        bchars[b] = chr(256+i)\n    charbytes = {c:b for b,c in bchars.items()}\n    tokenbytes = [b'',b'',b''] + [bytes(charbytes[c] for c in tok.id_to_token(i)) for i in range(8189)]\n    tokenbytes += [t.encode() for t in added]\n    assert all(tok.encode(t, add_special_tokens=False).ids == [8189+i] for i,t in enumerate(added))\n    return tok, tokenbytes\n\n", 'final_decode.py': '"""V3 decode loop; frozen Context/Decoder mathematics live in final_fast."""\nimport torch\nfrom torch.nn import functional as F\nimport final_core as C\n\n\n@torch.inference_mode()\ndef decode(context, k=1, temperature=None, deadline=None, check_every=16, compute_logprob=True):\n    from final_fast import Decoder\n    d = Decoder(context, k)\n    device = context.src.device\n    token = torch.full((d.n,), C.BOS, device=device, dtype=torch.long)\n    done = torch.zeros(d.n, dtype=torch.bool, device=device)\n    # KV, output, running LP/length buffers are allocated once, never concatenated.\n    output = torch.full((d.n, context.capacity), C.PAD, device=device, dtype=torch.long)\n    sums = torch.zeros(d.n, device=device)\n    lengths = torch.zeros(d.n, dtype=torch.long, device=device)\n    for t in range(context.capacity):\n        if t % check_every == 0:\n            C.check_deadline(deadline)\n        logits = d.step(token)\n        if temperature is not None or compute_logprob:\n            logits = logits.float()\n        logits[:, :2] = -torch.inf  # PAD/BOS: contiguous slice avoids a per-step index tensor.\n        pick = logits.argmax(-1) if temperature is None else torch.multinomial(\n            F.softmax(logits / temperature, -1), 1).squeeze(-1)\n        if compute_logprob:\n            logprob = F.log_softmax(logits, -1).gather(-1, pick[:, None]).squeeze(-1)\n            sums.add_(torch.where(done, 0., logprob))\n        lengths.add_(~done)\n        token = pick.masked_fill(done, C.PAD)\n        output[:, t] = token\n        done |= token.eq(C.EOS)\n        if (t + 1) % check_every == 0 and bool(done.all().cpu()):\n            break\n    return (output[:, :d.t].reshape(len(context.src), k, d.t).cpu(),\n            (sums / lengths.clamp_min(1)).reshape(-1, k).cpu(), d.cross_bytes)\n', 'final_numeric.py': '"""Final-only numeric policy; historical experiment executors remain unchanged."""\nimport ast\nimport builtins\nimport math\nimport multiprocessing as mp\nimport os\nimport random\nimport resource\nimport signal\nimport sys\nimport time\nfrom collections import Counter, deque\nfrom types import SimpleNamespace\n\nSAFE_BUILTINS = {n: getattr(builtins, n) for n in (\n    \'abs\',\'all\',\'any\',\'bool\',\'dict\',\'divmod\',\'enumerate\',\'float\',\'int\',\'len\',\'list\',\n    \'max\',\'min\',\'pow\',\'range\',\'reversed\',\'round\',\'set\',\'sorted\',\'str\',\'sum\',\'tuple\',\'zip\',\n    \'Exception\',\'ValueError\',\'TypeError\',\'ZeroDivisionError\',\'OverflowError\',\'ArithmeticError\')}\nFORBIDDEN_NAMES = {\'open\',\'input\',\'eval\',\'exec\',\'compile\',\'globals\',\'locals\',\'vars\',\'getattr\',\n    \'setattr\',\'delattr\',\'dir\',\'help\',\'breakpoint\',\'exit\',\'quit\',\'os\',\'sys\',\'subprocess\',\n    \'socket\',\'ctypes\',\'importlib\',\'builtins\',\'resource\',\'signal\',\'pickle\',\'marshal\',\n    \'memoryview\',\'type\',\'object\',\'super\'}\nFORBIDDEN_ATTRS = {\'mro\',\'subclasses\',\'format_map\',\'gi_frame\',\'f_globals\',\'f_locals\',\n    \'tb_frame\',\'func_globals\',\'code\',\'modules\',\'system\',\'popen\',\'load\',\'loads\',\'dump\',\'dumps\',\'random\',\'format\'}\nSAFE_BUILTINS[\'print\']=lambda *args,**kwargs:None  # no program output or file handle\nALLOWED_IMPORTS = {\'math\',\'sympy\'}\nSYM_NAMES = {\'Symbol\',\'symbols\',\'Rational\',\'Integer\',\'Float\',\'Eq\',\'solve\',\'sqrt\',\'simplify\'}\nMAX_CODE_BYTES=100000\nMAX_AST_NODES=5000\n\nclass ProgramTimeout(Exception): pass\n\ndef filter_program(code):\n    if not isinstance(code,str) or len(code.encode(\'utf-8\'))>MAX_CODE_BYTES:\n        return None,\'blocked_size\'\n    try: tree=ast.parse(code)\n    except (SyntaxError,ValueError,RecursionError): return None,\'syntax_error\'\n    nodes=list(ast.walk(tree))\n    if len(nodes)>MAX_AST_NODES:return None,\'blocked_size\'\n    for n in nodes:\n        if isinstance(n,ast.Attribute) and isinstance(n.ctx,(ast.Store,ast.Del)):\n            return None,\'blocked_attribute_mutation\'\n        if isinstance(n,ast.Name) and (n.id.startswith(\'__\') or n.id in FORBIDDEN_NAMES):\n            return None,\'blocked_name\'\n        if isinstance(n,ast.Attribute) and (n.attr.startswith(\'_\') or n.attr in FORBIDDEN_ATTRS):\n            return None,\'blocked_attribute\'\n        if isinstance(n,(ast.ClassDef,ast.AsyncFunctionDef,ast.Await,ast.Yield,ast.YieldFrom)):\n            return None,\'blocked_construct\'\n        if isinstance(n,(ast.FunctionDef,ast.arg)):\n            name=n.name if isinstance(n,ast.FunctionDef) else n.arg\n            if name.startswith(\'__\') or name in FORBIDDEN_NAMES:return None,\'blocked_name\'\n        if isinstance(n,ast.Import):\n            if any(a.name not in ALLOWED_IMPORTS or (a.asname and a.asname.startswith(\'_\'))for a in n.names):\n                return None,\'blocked_import\'\n        if isinstance(n,ast.ImportFrom):\n            if n.level or n.module not in ALLOWED_IMPORTS:return None,\'blocked_import\'\n            if any(a.name.startswith(\'_\') or a.name==\'*\'or (a.asname and a.asname.startswith(\'_\'))for a in n.names):\n                return None,\'blocked_import\'\n            if n.module==\'sympy\' and any(a.name not in SYM_NAMES for a in n.names):return None,\'blocked_import\'\n    return tree,None\n\ndef safe_sympy():\n    import sympy\n    # All sympify bindings import this parser dynamically, including object methods.\n    # Symbol names use the separately validated symbols() syntax, never parse_expr.\n    from sympy.parsing import sympy_parser\n    def denied_parser(*args,**kwargs):\n        raise ValueError(\'String expression parsing forbidden\')\n    sympy_parser.parse_expr=denied_parser\n    from sympy.core.cache import clear_cache\n    clear_cache()\n    from sympy.core.random import seed\n    seed(0)\n    def no_strings(x):\n        if isinstance(x,str):raise ValueError(\'String expressions forbidden\')\n        if isinstance(x,(tuple,list)):\n            for y in x:no_strings(y)\n        if isinstance(x,dict):\n            for k,v in x.items():no_strings(k);no_strings(v)\n    def wrapped(function):\n        def call(*args,**kwargs):\n            no_strings(args);no_strings(list(kwargs.values()))\n            return function(*args,**kwargs)\n        return call\n    def symbol(name,*args,**kwargs):\n        if not isinstance(name,str) or not name or not name.replace(\'_\',\'\').isalnum():raise ValueError(\'Bad symbol\')\n        return sympy.Symbol(name,*args,**kwargs)\n    def symbols(names,*args,**kwargs):\n        if not isinstance(names,str)or any(not(c.isalnum()or c in \' _,:\')for c in names):raise ValueError(\'Bad symbols\')\n        return sympy.symbols(names,*args,**kwargs)\n    return SimpleNamespace(Symbol=symbol,symbols=symbols,\n        **{n:wrapped(getattr(sympy,n))for n in SYM_NAMES-{\'Symbol\',\'symbols\'}})\n\n_MODULES={}\ndef safe_import(name,globals=None,locals=None,fromlist=(),level=0):\n    if level or name not in ALLOWED_IMPORTS:raise ImportError(\'Blocked import\')\n    if name not in _MODULES:\n        _MODULES[name]=SimpleNamespace(**{n:getattr(math,n)for n in dir(math)if not n.startswith(\'_\')})if name==\'math\'else safe_sympy()\n    return _MODULES[name]\n\ndef worker_init(memory_bytes):\n    # Spawned workers import no Torch. No GPU state or model is inherited.\n    resource.setrlimit(resource.RLIMIT_CORE,(0,0))\n    if sys.platform.startswith(\'linux\') and hasattr(resource,\'RLIMIT_AS\'):\n        soft,hard=resource.getrlimit(resource.RLIMIT_AS)\n        value=memory_bytes if hard==resource.RLIM_INFINITY else min(memory_bytes,hard)\n        resource.setrlimit(resource.RLIMIT_AS,(value,value))\n    signal.signal(signal.SIGALRM,lambda signum,frame:(_ for _ in ()).throw(ProgramTimeout()))\n\ndef prepare_pristine_worker():\n    """Run once in a Torch-free, single-threaded parent; never run user code here."""\n    _MODULES.clear()\n    safe_import(\'math\')\n    safe_import(\'sympy\')  # Parser deny patch, cache clear and SymPy seed happen ONCE.\n    random.seed(0)\n\ndef execute_task(payload, tree=None, isolated=False):\n    code,wall_limit,cpu_limit=payload\n    started=time.monotonic()\n    if not isolated:\n        _MODULES.clear()  # Historical/reference path; fork children inherit pristine modules.\n    random.seed(0)\n    if tree is None:\n        tree,status=filter_program(code)\n        if status:return {\'status\':status}\n    # RLIMIT_CPU is cumulative per process. Set a new SOFT deadline for EACH task;\n    # retain the hard limit so the soft deadline can increase for reused workers.\n    used=resource.getrusage(resource.RUSAGE_SELF)\n    soft=max(1,math.ceil(used.ru_utime+used.ru_stime)+int(math.ceil(cpu_limit)))\n    _,hard=resource.getrlimit(resource.RLIMIT_CPU)\n    if hard!=resource.RLIM_INFINITY:soft=min(soft,hard)\n    resource.setrlimit(resource.RLIMIT_CPU,(soft,hard))\n    signal.signal(signal.SIGPROF,lambda *_:(_ for _ in ()).throw(ProgramTimeout()))\n    signal.setitimer(signal.ITIMER_PROF,cpu_limit)\n    signal.setitimer(signal.ITIMER_REAL,wall_limit)\n    try:\n        env={\'__builtins__\':dict(SAFE_BUILTINS,__import__=safe_import)}\n        exec(compile(tree,\'<bounded-dataset-program>\',\'exec\'),env,env)\n        solve=env.get(\'solve\')\n        if not callable(solve):return {\'status\':\'missing_solve\'}\n        value=solve()\n        if isinstance(value,bool):value=int(value)\n        if not isinstance(value,(int,float)):\n            # SymPy real scalars may be used; containers/strings/complex refused.\n            if type(value).__module__.startswith(\'sympy\') and value.is_real is True:value=float(value)\n            else:return {\'status\':\'non_number\'}\n        value=float(value)\n        if not math.isfinite(value):return {\'status\':\'non_finite\'}\n        return {\'status\':\'numeric\',\'value\':value}\n    except ProgramTimeout:return {\'status\':\'timeout\'}\n    except BaseException as e:return {\'status\':\'execution_error\',\'error_type\':type(e).__name__}\n    finally:\n        signal.setitimer(signal.ITIMER_REAL,0)\n        signal.setitimer(signal.ITIMER_PROF,0)\n', 'final_vote.py': '"""Frozen mixed vote. final_choose is generated verbatim from exp27_decode.choose."""\nimport json\nimport math\nfrom pathlib import Path\nfrom final_choose import choose\n\nCONFIG = {\'mode\': \'vote\', \'temperature\': .7, \'k\': 8, \'greedy_vote\': True}\n\n\ndef select(greedy, main, long):\n    # Deduplication only caches execution: repeated samples still cast votes.\n    samples = [c for c in main[:4] + long[:4] if c.get(\'mean_logprob\') is not None\n               and math.isfinite(c[\'mean_logprob\'])]\n    row = {\'greedy\': greedy, \'samples\': {\'0.7\': samples}}\n    winner = choose(row, CONFIG)\n    if winner[\'execution\'][\'status\'] != \'numeric\':\n        return greedy\n    # Code representative: greedy anywhere in the winning connected component,\n    # otherwise greatest mean LP, stable order. Do not replace graph connectivity\n    # by a direct isclose comparison (isclose is non-transitive).\n    good = [c for c in [greedy] + samples if c[\'execution\'][\'status\'] == \'numeric\']\n    component = [winner]\n    remaining = [c for c in good if c is not winner]\n    while True:\n        adjacent = [c for c in remaining if any(math.isclose(\n            c[\'execution\'][\'value\'], v[\'execution\'][\'value\'], rel_tol=1e-6, abs_tol=1e-6)\n            for v in component)]\n        if not adjacent:\n            break\n        component.extend(adjacent)\n        adjacent_ids = {id(c) for c in adjacent}\n        remaining = [c for c in remaining if id(c) not in adjacent_ids]\n    if any(c is greedy for c in component):\n        return greedy\n    return winner\n\n\ndef select_available(record):\n    """A missing model/batch never discards already executed samples or their votes."""\n    available = {key: [c for c in record[key]\n                       if c[\'execution\'][\'status\'] != \'budget_not_evaluated\']\n                 for key in (\'main\', \'long\')}\n    count = len(available[\'main\']) + len(available[\'long\'])\n    return select(record[\'greedy\'], available[\'main\'], available[\'long\']), count\n\n\ndef write_answers(path, codes):\n    with Path(path).open(\'w\', encoding=\'utf8\') as stream:\n        for code in codes:\n            if not isinstance(code, str):\n                raise TypeError(\'Answer code must be a string\')\n            stream.write(json.dumps({\'code\': code}, ensure_ascii=False) + \'\\n\')\n\n\ndef validate_answers(path, expected):\n    with Path(path).open(encoding=\'utf8\') as stream:\n        rows = [json.loads(line) for line in stream]\n    assert len(rows) == expected\n    assert all(set(r) == {\'code\'} and isinstance(r[\'code\'], str) for r in rows)\n    return rows\n\n\ndef verify_winners(winners, deadline, workers=None):\n    """Fresh service: compare the actual chosen CODE against its voted execution."""\n    from final_executor import Executor\n    executor = Executor(workers)\n    eligible = [i for i, w in enumerate(winners) if w[\'execution\'][\'status\'] == \'numeric\']\n    items = [{\'code\': winners[i][\'code\']} for i in eligible]\n    try:\n        executor.submit(items)\n        complete = executor.drain(deadline)\n        executor.attach(items)\n        mismatches = [i for i, item in zip(eligible, items) if\n                      item[\'execution\'][\'status\'] != \'numeric\' or not math.isclose(\n                          item[\'execution\'][\'value\'], winners[i][\'execution\'][\'value\'],\n                          rel_tol=1e-6, abs_tol=1e-6)]\n        return {\'checked_rows\': len(eligible), \'complete\': complete,\n                \'mismatch_rows\': mismatches, \'execution\': executor.summary()}\n    finally:\n        executor.close()\n', 'final_execute.py': '"""Persistent Torch-free service: fresh numeric namespaces, pristine SymPy forks.\n\nstdin: JSON lists of [id, code], or null to finish. stdout: [id, execution, metrics].\nThe launcher can kill the complete service process group, including its workers.\n"""\nimport json\nimport multiprocessing as mp\nimport os\nimport queue\nimport resource\nimport select\nimport signal\nimport sys\nimport threading\nimport time\nfrom collections import deque\nfrom multiprocessing.connection import wait\nimport final_numeric as numeric\n\nMEMORY = 768 * 1024**2\nWALL = 3.\nCPU = 2.\n\n\ndef kill_program(pid):\n    # Each program owns a group; kill descendants even after a successful result.\n    try:\n        os.killpg(pid, signal.SIGKILL)\n    except (ProcessLookupError, PermissionError):\n        try:\n            os.kill(pid, signal.SIGKILL)  # Child may not yet have called setsid().\n        except ProcessLookupError:\n            pass\n    until = time.monotonic()+.1\n    while time.monotonic() < until:\n        try:\n            if os.waitpid(pid, os.WNOHANG)[0]:\n                return\n        except ChildProcessError:\n            return\n        time.sleep(.001)\n    raise RuntimeError(\'Program could not be reaped; replace pristine worker\')\n\n\ndef isolated_task(code, tree, connection):\n    reader, writer = os.pipe()\n    parent = os.getpid()\n    pid = os.fork()  # This parent has no Torch/CUDA and no reader/BLAS threads.\n    if pid == 0:\n        try:\n            os.close(reader)\n            connection.close()\n            os.setsid()\n            if sys.platform.startswith(\'linux\'):\n                import ctypes\n                signal.signal(signal.SIGTERM, lambda *_: os.killpg(os.getpgrp(), signal.SIGKILL))\n                if ctypes.CDLL(None).prctl(1, signal.SIGTERM, 0, 0, 0) != 0 or os.getppid() != parent:\n                    os.killpg(os.getpgrp(), signal.SIGKILL)\n            resource.setrlimit(resource.RLIMIT_CPU, (int(CPU)+1, int(CPU)+1))\n            result = numeric.execute_task((code, WALL, CPU), tree=tree, isolated=True)\n            os.write(writer, json.dumps(result, separators=(\',\', \':\')).encode())\n        finally:\n            os._exit(0)\n    os.close(writer)\n    try:\n        # Same outer limit as v2; catches code that repeatedly catches timer exceptions.\n        if select.select([reader], [], [], WALL+2.)[0]:\n            data = os.read(reader, 4096)\n            if data:\n                return json.loads(data)\n        return {\'status\': \'timeout_or_worker_exit\'}\n    finally:\n        os.close(reader)\n        kill_program(pid)\n\n\ndef worker(connection):\n    # Library diagnostics and user output cannot corrupt the JSON protocol.\n    with open(os.devnull, \'w\') as sink:\n        os.dup2(sink.fileno(), 1)\n        os.dup2(sink.fileno(), 2)\n    numeric.worker_init(MEMORY)\n    pristine = None\n    while True:\n        item = connection.recv()\n        if item is None:\n            return\n        index, code = item\n        started = time.monotonic()\n        used = time.process_time()\n        before = resource.getrusage(resource.RUSAGE_CHILDREN)\n        tree, status = numeric.filter_program(code)\n        if status:\n            result = {\'status\': status}\n        elif any((isinstance(node, numeric.ast.Import) and any(a.name == \'sympy\' for a in node.names)) or\n                 (isinstance(node, numeric.ast.ImportFrom) and node.module == \'sympy\')\n                 for node in numeric.ast.walk(tree)):\n            if pristine is None:\n                numeric.prepare_pristine_worker()\n                pristine = dict(numeric._MODULES)\n            numeric._MODULES.clear()\n            numeric._MODULES.update(pristine)\n            result = isolated_task(code, tree, connection)\n        else:\n            # Only builtins and a fresh math namespace are reachable on this path.\n            # No user code can access SymPy objects/caches; env and timers are per task.\n            result = numeric.execute_task((code, WALL, CPU), tree=tree)\n        after = resource.getrusage(resource.RUSAGE_CHILDREN)\n        connection.send((index, result, {\'task_wall_seconds\': time.monotonic()-started,\n            \'task_cpu_seconds\': time.process_time()-used + after.ru_utime+after.ru_stime-before.ru_utime-before.ru_stime}))\n\n\ndef serve(workers):\n    inbox = queue.Queue()\n    def read():\n        for line in sys.stdin:\n            inbox.put(json.loads(line))\n        inbox.put(None)\n    threading.Thread(target=read, daemon=True).start()\n    context = mp.get_context(\'spawn\')\n    slots = []\n    pending = deque()\n    finished = False\n    def start():\n        parent, child = context.Pipe()\n        process = context.Process(target=worker, args=(child,), daemon=True)\n        process.start()\n        child.close()\n        return [process, parent, None, 0.]\n    def stop(slot):\n        process, pipe, _, _ = slot\n        if process.is_alive():\n            process.kill()\n        process.join(timeout=.05)\n        pipe.close()\n    def emit(index, result, metrics=None):\n        print(json.dumps([index, result, metrics]), flush=True)\n    try:\n        slots = [start() for _ in range(workers)]\n        while not finished or pending or any(s[2] is not None for s in slots):\n            while True:\n                try:\n                    batch = inbox.get_nowait()\n                except queue.Empty:\n                    break\n                if batch is None:\n                    finished = True\n                else:\n                    pending.extend(batch)\n            for number, slot in enumerate(slots):\n                process, pipe, active, submitted = slot\n                pipe_failed = False\n                # Worker enforces wall+2; allow a bounded IPC/reap margin here.\n                try:\n                    if active is not None and pipe.poll():\n                        index, result, *metadata = pipe.recv()\n                        assert index == active\n                        emit(index, result, metadata[0] if metadata else None)\n                        slot[2] = None\n                except (EOFError, OSError):\n                    pipe_failed = True\n                if pipe_failed or not process.is_alive() or (slot[2] is not None and time.monotonic()-submitted >= WALL+2.5):\n                    if slot[2] is not None:\n                        emit(slot[2], {\'status\': \'timeout_or_worker_exit\'})\n                    stop(slot)\n                    slots[number] = slot = start()\n                    process, pipe = slot[:2]\n                if slot[2] is None and pending:\n                    index, code = pending.popleft()\n                    try:\n                        pipe.send((index, code))\n                        slot[2:] = [index, time.monotonic()]\n                    except (EOFError, OSError):\n                        # Death can race is_alive(): this job was not accepted.\n                        pending.appendleft((index, code))\n                        stop(slot)\n                        slots[number] = start()\n            active_pipes = [slot[1] for slot in slots if slot[2] is not None]\n            if active_pipes:\n                wait(active_pipes, timeout=.002)  # Wake immediately, no per-program 1ms polling tax.\n            else:\n                time.sleep(.002)\n    finally:\n        for slot in slots:\n            stop(slot)\n\n\nif __name__ == \'__main__\':\n    # Independent supervisor kills the process group on parent death (Linux).\n    if sys.platform.startswith(\'linux\'):\n        import ctypes\n        parent = os.getppid()\n        signal.signal(signal.SIGTERM, lambda *_: os.killpg(os.getpgrp(), signal.SIGKILL))\n        assert ctypes.CDLL(None).prctl(1, signal.SIGTERM, 0, 0, 0) == 0\n        if parent == 1 or os.getppid() != parent:\n            os.killpg(os.getpgrp(), signal.SIGKILL)\n    serve(int(sys.argv[1]))\n', 'final_executor.py': '"""Incremental global execution cache, usable while the GPU generates next batch."""\nimport json\nimport os\nimport signal\nimport subprocess\nimport sys\nimport threading\nimport time\nfrom pathlib import Path\n\n\nclass Executor:\n    def __init__(self, workers=None, service_path=None):\n        self.workers = workers if workers is not None else (os.cpu_count() or 1)\n        env = dict(os.environ, OMP_NUM_THREADS=\'1\', MKL_NUM_THREADS=\'1\', OPENBLAS_NUM_THREADS=\'1\',\n                   PYTHONHASHSEED=\'0\')\n        self.child = subprocess.Popen([sys.executable, str(service_path or Path(__file__).with_name(\'final_execute.py\')),\n                                      str(self.workers)], stdin=subprocess.PIPE, stdout=subprocess.PIPE,\n                                      stderr=subprocess.DEVNULL, text=True, bufsize=1, env=env,\n                                      start_new_session=True)\n        self.codes = {}\n        self.results = {}\n        self.requests = 0\n        self.started = time.monotonic()\n        self.error = None\n        self.task_wall_seconds = self.task_cpu_seconds = 0.\n        self.first_submit = self.last_result = None\n        def read():\n            try:\n                for line in self.child.stdout:\n                    index, value, *metadata = json.loads(line)\n                    if metadata and metadata[0]:\n                        self.task_wall_seconds += metadata[0][\'task_wall_seconds\']\n                        self.task_cpu_seconds += metadata[0][\'task_cpu_seconds\']\n                    self.last_result = time.monotonic()\n                    self.results[index] = value\n            except Exception as error:\n                self.error = type(error).__name__\n        self.reader = threading.Thread(target=read, daemon=True)\n        self.reader.start()\n\n    def submit(self, candidates):\n        jobs = []\n        for candidate in candidates:\n            self.requests += 1\n            code = candidate[\'code\']\n            if code not in self.codes:\n                index = len(self.codes)\n                self.codes[code] = index\n                jobs.append([index, code])\n        if jobs:\n            if self.first_submit is None:\n                self.first_submit = time.monotonic()\n            self.child.stdin.write(json.dumps(jobs) + \'\\n\')\n            self.child.stdin.flush()\n\n    def attach(self, candidates):\n        for candidate in candidates:\n            index = self.codes[candidate[\'code\']]\n            candidate[\'execution\'] = self.results.get(index, {\'status\': \'budget_not_evaluated\'})\n\n    def drain(self, deadline):\n        while len(self.results) < len(self.codes) and time.monotonic() < deadline:\n            exit_code = self.child.poll()\n            if exit_code is not None:\n                self.error = self.error or \'executor_exit_\'+str(exit_code)\n                break\n            if self.error:\n                break\n            time.sleep(.002)\n        return len(self.results) == len(self.codes)\n\n    def summary(self):\n        return {\'workers\': self.workers, \'requested_candidates\': self.requests,\n                \'unique_programs\': len(self.codes), \'deduplicated\': self.requests-len(self.codes),\n                \'completed_unique\': len(self.results), \'service_wall_seconds\': time.monotonic()-self.started,\n                \'sum_worker_task_wall_seconds\': self.task_wall_seconds,\n                \'sum_worker_task_cpu_seconds\': self.task_cpu_seconds,\n                \'first_submit_to_last_result_seconds\': (self.last_result-self.first_submit\n                    if self.last_result is not None and self.first_submit is not None else None),\n                \'service_wall_includes_GPU_wait\': True,\n                \'wall_limit\': 3., \'cpu_limit\': 2., \'memory_bytes\': 768*1024**2,\n                \'overlap\': True, \'isolation\': \'fresh numeric namespaces / pristine SymPy fork\', \'error\': self.error}\n\n    def close(self):\n        # Never pool.join() unbounded in the CUDA process.\n        try:\n            os.killpg(self.child.pid, signal.SIGKILL)\n        except ProcessLookupError:\n            pass\n        try:\n            self.child.wait(timeout=.1)\n        except subprocess.TimeoutExpired:\n            pass\n        self.child.stdin.close()\n        self.reader.join(timeout=.1)\n        self.child.stdout.close()\n', 'final_inference.py': '"""Cold mixed C-main/C-long2x inference on one GPU; no training or targets in selection."""\nimport os\nimport time\nPROCESS_START = float(os.environ.get(\'FINAL_PROCESS_START\', time.monotonic()))\nimport sys\nimport signal\n# Arm before Torch import: session watchdog can kill the supervisor and its entire tree.\nif __name__ == \'__main__\' and sys.platform.startswith(\'linux\'):\n    import ctypes\n    _parent = os.getppid()\n    signal.signal(signal.SIGTERM, lambda *_: os.killpg(os.getpgrp(), signal.SIGKILL))\n    assert ctypes.CDLL(None).prctl(1, signal.SIGTERM, 0, 0, 0) == 0\n    if _parent == 1 or os.getppid() != _parent:\n        os.killpg(os.getpgrp(), signal.SIGKILL)\nimport argparse\nimport hashlib\nimport json\nimport math\nfrom pathlib import Path\nimport torch\nfrom torch import nn\nimport final_core as C\nimport final_fast as Fast\nfrom final_representation import tokenizer, restore_compact\nfrom final_executor import Executor\nfrom final_vote import select_available, write_answers, verify_winners\n\nWEIGHT_SHA = {\n    \'C-main\': \'3806a5827004fd88f533a7307abc0c1e387faafc5672f4224392b363c4d2ad3d\',\n    \'C-long2x\': \'6227ee2fc8dd12e186137be229fef2b1ceba659848c35bedd90cec6502f41548\'}\n\n\ndef sha(path):\n    h = hashlib.sha256()\n    with Path(path).open(\'rb\') as stream:\n        for block in iter(lambda: stream.read(4*1024**2), b\'\'):\n            h.update(block)\n    return h.hexdigest()\n\n\ndef write(path, value):\n    temporary = Path(path).with_suffix(\'.tmp\')\n    temporary.write_text(json.dumps(value, ensure_ascii=False, indent=2)+\'\\n\')\n    temporary.replace(path)\n\n\ndef load(path, device):\n    payload = torch.load(path, map_location=\'cpu\', weights_only=True)\n    assert payload[\'format\'] == \'nlp-hw02-model-only-v1\' and payload[\'optimizer_included\'] is False\n    model = C.SimpleTransformer(**payload[\'model_spec\'], dropout=0.).eval()\n    if device.type == \'cuda\':\n        model.half()\n    model.load_state_dict(payload[\'model\'], strict=True)\n    return model.to(device)\n\n\ndef pack(ids, lps, table, greedy=False):\n    groups = []\n    for sequences, means in zip(ids.tolist(), lps.tolist()):\n        group = []\n        for tokens, lp in zip(sequences, means):\n            active = C.active_tokens(tokens)\n            represented = C.decode_bytes(table, active).decode(\'utf8\', errors=\'replace\')\n            try:\n                code, error = restore_compact(represented), None\n            except (ValueError, IndexError):\n                code, error = \'\', \'invalid representation\'\n            group.append({\'code\': code, \'mean_logprob\': 0. if greedy else (lp if math.isfinite(lp) else None),\n                          \'represented_code\': represented, \'restore_error\': error, \'length\': len(active)})\n        groups.append(group)\n    return groups\n\n\ndef run(config):\n    phases = {\'imports_and_launcher\': time.monotonic()-PROCESS_START}\n    out = Path(config[\'output\']); out.mkdir(parents=True, exist_ok=True)\n    budget = float(config.get(\'budget_seconds\', 180.))\n    # GPU/CPU work stops well before the external hard stop, leaving voting/write reserve.\n    work_deadline = PROCESS_START + budget - min(40., budget*.23)\n    execution_deadline = PROCESS_START + budget - min(28., budget*.16)\n    verification_deadline = PROCESS_START + budget - min(10., budget*.06)\n    greedy_deadline = PROCESS_START + budget - min(50., budget*.28)\n    started = time.monotonic()\n    rows = json.loads(Path(config[\'rows\']).read_text())\n    phases[\'read_input\'] = time.monotonic()-started\n    assert rows and all(isinstance(r[\'question\'], str) for r in rows)\n    if config.get(\'expected_rows\') is not None:\n        assert len(rows) == config[\'expected_rows\']\n    device = torch.device(config.get(\'device\', \'cuda:0\'))\n    if device.type == \'cpu\' and config.get(\'cpu_test_expire_sampling\'):\n        work_deadline = PROCESS_START  # Deterministic CPU-only guard fixture.\n    torch.set_num_threads(1)\n    torch.set_num_interop_threads(1)\n    torch.backends.cuda.matmul.allow_tf32 = False\n    torch.use_deterministic_algorithms(True)\n    # The decoder writes each KV slot before reading it; avoid filling unused slots.\n    torch.utils.deterministic.fill_uninitialized_memory = False\n    torch.backends.cudnn.benchmark = False\n    torch.backends.cudnn.deterministic = True\n    if device.type == \'cuda\':\n        assert os.environ.get(\'CUDA_VISIBLE_DEVICES\') == \'0\' and torch.cuda.device_count() == 1\n        assert \'T4\' in torch.cuda.get_device_name(0)\n        assert str(torch.__version__) == \'2.11.0+cu128\' and torch.version.cuda == \'12.8\', \'Pinned L2 runtime required\'\n    started = time.monotonic()\n    if config.get(\'verify_weights\', True):\n        for name, path in config[\'models\'].items():\n            assert sha(path) == WEIGHT_SHA[name], name\n    if config.get(\'tokenizer_sha\'):\n        assert sha(config[\'tokenizer\']) == config[\'tokenizer_sha\']\n    models = {name: load(path, device) for name, path in config[\'models\'].items()}\n    tok, table = tokenizer(config[\'tokenizer\'], 30)\n    encoded = [[C.BOS]+C.encode(tok, r[\'question\'])+[C.EOS] for r in rows]\n    phases[\'load_models_tokenizer_encode\'] = time.monotonic()-started\n    started = time.monotonic()\n    # No stochastic warmup; seed is reset separately for each model afterwards.\n    for model in models.values():\n        src = torch.tensor([encoded[0]], device=device)\n        ctx = Fast.Context(model, src, 16)\n        Fast.decode(ctx)\n        del ctx, src\n    C.sync(device)\n    phases[\'warmup\'] = time.monotonic()-started\n    executor = Executor(config.get(\'workers\'))\n    records = [{\'greedy\': None, \'main\': [], \'long\': []} for _ in rows]\n    guard_events = []\n    cap = int(config.get(\'cap\', 512))\n    restore_seconds = 0.\n    attempts = []\n    batch_stats = []\n\n    def generate(name, order, batch, k, temperature, target, deadline):\n        nonlocal restore_seconds\n        offset = 0\n        while offset < len(order):\n            C.check_deadline(deadline)\n            selected = order[offset:offset+batch]\n            src = ctx = seq = means = None\n            rng_cpu = torch.get_rng_state() if temperature is not None else None\n            rng_gpu = torch.cuda.get_rng_state(device) if temperature is not None and device.type == \'cuda\' else None\n            had_oom = False\n            try:\n                src = nn.utils.rnn.pad_sequence([torch.tensor(encoded[i]) for i in selected],\n                    batch_first=True, padding_value=C.PAD).to(device)\n                ctx = Fast.Context(models[name], src, cap)\n                seq, means, _ = Fast.decode(ctx, k, temperature, deadline=deadline,\n                                           compute_logprob=temperature is not None)\n                begin = time.monotonic()\n                groups = pack(seq, means, table, greedy=temperature is None)\n                restore_seconds += time.monotonic()-begin\n                for i, group in zip(selected, groups):\n                    records[i][target] = group[0] if target == \'greedy\' else group[:4]\n                executor.submit([c for group in groups for c in group[:(1 if target == \'greedy\' else 4)]])\n                batch_stats.append({\'phase\': target, \'questions\': len(selected), \'sequences\': len(selected)*k,\n                                    \'source_tokens\': src.shape[1], \'decode_steps\': seq.shape[-1]})\n                offset += len(selected)\n                if (device.type == \'cpu\' and target == \'main\' and\n                    config.get(\'cpu_test_expire_after_main_batches\') == sum(s[\'phase\'] == \'main\' for s in batch_stats)):\n                    raise C.BudgetExpired(\'CPU partial-candidate guard fixture\')\n            except torch.cuda.OutOfMemoryError:\n                had_oom = True\n                if rng_cpu is not None:\n                    torch.set_rng_state(rng_cpu)\n                if rng_gpu is not None:\n                    torch.cuda.set_rng_state(rng_gpu, device)\n                attempts.append({\'model\': name, \'offset\': offset, \'question_batch\': batch, \'K\': k})\n                batch //= 2\n                if batch < 1:\n                    raise\n            finally:\n                del src, ctx, seq, means\n                if had_oom:\n                    # Only OOM requires collection. Normal batches reuse the CUDA allocator.\n                    import gc\n                    gc.collect()\n                    if device.type == \'cuda\':\n                        torch.cuda.empty_cache()\n\n    try:\n        started = time.monotonic()\n        greedy_order = sorted(range(len(rows)), key=lambda i: (len(encoded[i]), i))\n        generate(\'C-main\', greedy_order, config.get(\'greedy_batch\', 1024), 1, None, \'greedy\', greedy_deadline)\n        phases[\'greedy_generation_restore_submit\'] = time.monotonic()-started\n        # Completed before sampling: the independent supervisor can publish it after a hard kill.\n        started = time.monotonic()\n        write_answers(out/\'greedy-fallback.jsonl\', [r[\'greedy\'][\'code\'] for r in records])\n        write(out/\'greedy-ready.json\', {\'rows\': len(rows), \'seconds\': time.monotonic()-PROCESS_START})\n        phases[\'greedy_checkpoint_write\'] = time.monotonic()-started\n        replay = config.get(\'l2_replay\', False)\n        # Length buckets reduce source padding; writeback always uses the original row ID.\n        # Only the diagnostic L2 replay retains its original order and batch32/K8 RNG lanes.\n        order = list(range(len(rows))) if replay else greedy_order\n        for name, target in [(\'C-main\', \'main\'), (\'C-long2x\', \'long\')]:\n            started = time.monotonic()\n            torch.manual_seed(270042)\n            if device.type == \'cuda\':\n                torch.cuda.manual_seed_all(270042)\n            try:\n                generate(name, order, 32 if replay else config.get(\'sample_batch\', 256),\n                         8 if replay else 4, .7, target, work_deadline)\n            except C.BudgetExpired:\n                guard_events.append({\'phase\': target, \'elapsed_seconds\': time.monotonic()-PROCESS_START})\n            phases[target+\'_sampling_restore_submit\'] = time.monotonic()-started\n            if guard_events:\n                break\n        started = time.monotonic()\n        drained = executor.drain(execution_deadline)\n        phases[\'execution_tail_wait\'] = time.monotonic()-started\n        if not drained:\n            guard_events.append({\'phase\': \'execution\', \'elapsed_seconds\': time.monotonic()-PROCESS_START})\n        candidates = [c for r in records for c in [r[\'greedy\']]+r[\'main\']+r[\'long\']]\n        executor.attach(candidates)\n        execution = executor.summary()\n    finally:\n        started = time.monotonic()\n        executor.close()\n        phases[\'executor_shutdown\'] = time.monotonic()-started\n    started = time.monotonic()\n    winners = []\n    fallback = []\n    partial_vote_rows = []\n    executed_sample_counts = []\n    for i, r in enumerate(records):\n        winner, count = select_available(r)\n        winners.append(winner)\n        executed_sample_counts.append(count)\n        if count == 0:\n            fallback.append(i)\n        elif count < 8:\n            partial_vote_rows.append(i)\n    phases[\'vote\'] = time.monotonic()-started\n    started = time.monotonic()\n    verification = verify_winners(winners, verification_deadline, config.get(\'workers\'))\n    for i in verification[\'mismatch_rows\']:\n        winners[i] = records[i][\'greedy\']\n        if i not in fallback:\n            fallback.append(i)\n    if verification[\'mismatch_rows\']:\n        guard_events.append({\'phase\': \'winner_reexecution\', \'rows\': verification[\'mismatch_rows\'],\n                             \'elapsed_seconds\': time.monotonic()-PROCESS_START})\n    phases[\'winner_reexecution\'] = time.monotonic()-started\n    started = time.monotonic()\n    write_answers(out/\'answers.jsonl\', [w[\'code\'] for w in winners])\n    closed = time.monotonic()\n    phases[\'answer_write\'] = closed-started\n    phases[\'bookkeeping_and_executor_start\'] = closed-PROCESS_START-sum(phases.values())\n    timing = {\'full_pipeline_seconds\': closed-PROCESS_START, \'rows\': len(rows), \'phases_seconds\': phases,\n              \'restore_seconds_in_generation\': restore_seconds, \'execution\': execution,\n              \'deadline_guard\': {\'triggered\': bool(guard_events), \'events\': guard_events,\n                                 \'fallback_rows\': sorted(fallback), \'fallback_count\': len(fallback)},\n              \'winner_reexecution\': verification,\n              \'no_numeric_winner_rows\': [i for i, w in enumerate(winners) if w[\'execution\'][\'status\'] != \'numeric\'],\n              \'input_sha256\': sha(config[\'rows\']),\n              \'question_order_sha256\': hashlib.sha256(json.dumps(\n                  [r[\'question\'] for r in rows], ensure_ascii=False).encode()).hexdigest(),\n              \'answers_sha256\': sha(out/\'answers.jsonl\'),\n              \'OOM_attempts\': attempts, \'single_gpu\': device.type == \'cuda\', \'seed\': 270042,\n              \'sample_K\': 8 if config.get(\'l2_replay\') else 4, \'used_K_per_model\': 4,\n              \'greedy_batch\': config.get(\'greedy_batch\', 1024),\n              \'sampling_batch\': 32 if config.get(\'l2_replay\') else config.get(\'sample_batch\', 256),\n              \'length_bucketing\': not config.get(\'l2_replay\', False), \'generation_batches\': batch_stats,\n              \'peak_reserved_bytes\': torch.cuda.max_memory_reserved(device) if device.type == \'cuda\' else None,\n              \'peak_allocated_bytes\': torch.cuda.max_memory_allocated(device) if device.type == \'cuda\' else None,\n              \'deterministic_algorithms\': True, \'python_hash_seed\': os.environ.get(\'PYTHONHASHSEED\'),\n              \'timer\': \'launcher process start through answers file close; imports/load/warmup/exec/vote/write included\'}\n    timing[\'deadline_guard\'].update(partial_vote_rows=partial_vote_rows,\n        partial_vote_count=len(partial_vote_rows), no_executed_sample_count=executed_sample_counts.count(0),\n        executed_sample_counts=executed_sample_counts,\n        generation_stop_seconds=work_deadline-PROCESS_START, execution_stop_seconds=execution_deadline-PROCESS_START,\n        verification_stop_seconds=verification_deadline-PROCESS_START)\n    write(out/\'timing.json\', timing)\n    # Diagnostic outputs and gold scoring are outside the answer timer, no gold is passed to select.\n    if config.get(\'reference\'):\n        reference = json.loads(Path(config[\'reference\']).read_text())\n        assert len(reference) == len(winners)\n        matches = [i for i, (ref, win) in enumerate(zip(reference, winners)) if\n                   ref[\'status\'] == win[\'execution\'][\'status\'] == \'numeric\' and math.isclose(\n                       ref[\'value\'], win[\'execution\'][\'value\'], rel_tol=1e-6, abs_tol=1e-6)]\n        write(out/\'quality.json\', {\'matches\': len(matches), \'n\': len(winners), \'accuracy_pp\': 100*len(matches)/len(winners),\n                                 \'match_indices\': matches, \'expected_matches\': 1492,\n                                 \'delta_pp\': 100*(len(matches)-1492)/len(winners),\n                                 \'tolerance_pp\': 1.,\n                                 \'passed\': abs(100*(len(matches)-1492)/len(winners)) <= 1. and not guard_events})\n        import gzip\n        with gzip.open(out/\'candidate-journal.json.gz\', \'wt\', encoding=\'utf8\') as stream:\n            json.dump(records, stream, ensure_ascii=False)\n    return timing\n\n\nif __name__ == \'__main__\':\n    parser = argparse.ArgumentParser()\n    parser.add_argument(\'config\')\n    run(json.loads(Path(parser.parse_args().config).read_text()))\n', 'final_runner.py': '"""Torch-free process supervisor. Hard stop and greedy publication share the honest timer."""\nimport time\nimport os\nSTART = float(os.environ.get(\'FINAL_PROCESS_START\', time.monotonic()))\nimport json\nimport signal\nimport subprocess\nimport sys\nfrom pathlib import Path\n\nif __name__ == \'__main__\' and sys.platform.startswith(\'linux\'):\n    import ctypes\n    _parent = os.getppid()\n    signal.signal(signal.SIGTERM, lambda *_: os.killpg(os.getpgrp(), signal.SIGKILL))\n    assert ctypes.CDLL(None).prctl(1, signal.SIGTERM, 0, 0, 0) == 0\n    if _parent == 1 or os.getppid() != _parent:\n        os.killpg(os.getpgrp(), signal.SIGKILL)\n\n\ndef supervise(config_file, start=None):\n    start = START if start is None else start\n    config = json.loads(Path(config_file).read_text())\n    out = Path(config[\'output\']); out.mkdir(parents=True, exist_ok=True)\n    old = [name for name in (\'answers.jsonl\', \'greedy-fallback.jsonl\', \'greedy-ready.json\',\n                             \'timing.json\', \'summary.json\', \'worker.log\') if (out/name).exists()]\n    if old:\n        raise ValueError(\'Fresh cold run requires a new output directory; existing artifacts: \'+\',\'.join(old))\n    budget = float(config.get(\'budget_seconds\', 180.))\n    env = dict(os.environ, CUDA_VISIBLE_DEVICES=\'0\', FINAL_PROCESS_START=str(start),\n               TOKENIZERS_PARALLELISM=\'false\', OMP_NUM_THREADS=\'1\', MKL_NUM_THREADS=\'1\', OPENBLAS_NUM_THREADS=\'1\',\n               PYTHONHASHSEED=\'0\', CUBLAS_WORKSPACE_CONFIG=\':4096:8\')\n    child = None\n    stopped = False\n    with (out/\'worker.log\').open(\'w\') as log:\n        child = subprocess.Popen([sys.executable, str(Path(__file__).with_name(\'final_inference.py\')), str(config_file)],\n                                 env=env, start_new_session=True, stdout=log, stderr=log)\n        try:\n            while child.poll() is None:\n                if time.monotonic() >= start+budget-min(8., budget*.045):\n                    stopped = True\n                    break\n                time.sleep(.01)\n        finally:\n            # Executor is a separate session; its Linux parent-death guard kills the tree.\n            try:\n                os.killpg(child.pid, signal.SIGKILL)\n            except ProcessLookupError:\n                pass\n            try:\n                child.wait(timeout=.1)\n            except subprocess.TimeoutExpired:\n                pass\n    timer_path = out/\'timing.json\'\n    if stopped or child.returncode or not timer_path.exists():\n        # Fatal GPU/driver/init failure before greedy completion cannot produce real greedy.\n        # Fail explicitly, never fabricate code or silently claim the time gate passed.\n        ready = out/\'greedy-ready.json\'\n        if not ready.exists():\n            report = {\'status\': \'failed_before_greedy\', \'elapsed_seconds\': time.monotonic()-start,\n                      \'exit_code\': child.returncode, \'hard_stop\': stopped}\n            (out/\'summary.json\').write_text(json.dumps(report, indent=2)+\'\\n\')\n            raise RuntimeError(\'Cold pipeline failed before greedy; inspect worker.log\')\n        data = (out/\'greedy-fallback.jsonl\').read_bytes()\n        with (out/\'answers.jsonl\').open(\'wb\') as stream:\n            stream.write(data)\n        closed = time.monotonic()\n        timing = {\'full_pipeline_seconds\': closed-start, \'rows\': json.loads(ready.read_text())[\'rows\'],\n                  \'deadline_guard\': {\'triggered\': True, \'supervisor\': True, \'fallback_rows\': \'all\',\n                                     \'fallback_count\': json.loads(ready.read_text())[\'rows\']},\n                  \'phases_seconds\': {}, \'worker_exit_code\': child.returncode, \'hard_stop\': stopped}\n        timer_path.write_text(json.dumps(timing, indent=2)+\'\\n\')\n    else:\n        timing = json.loads(timer_path.read_text())\n    timing[\'status\'] = \'guarded\' if timing[\'deadline_guard\'][\'triggered\'] else \'complete\'\n    timing[\'under_180_seconds\'] = timing[\'full_pipeline_seconds\'] < 180.\n    timing[\'target_150_seconds\'] = timing[\'full_pipeline_seconds\'] <= 150.\n    timing[\'target_120_seconds\'] = timing[\'full_pipeline_seconds\'] <= 120.\n    (out/\'summary.json\').write_text(json.dumps(timing, indent=2)+\'\\n\')\n    assert timing[\'full_pipeline_seconds\'] < budget, \'Budget exceeded: reject this run\'\n    return timing\n\n\nif __name__ == \'__main__\':\n    report = supervise(sys.argv[1])\n    print(json.dumps({k: report[k] for k in [\'status\', \'rows\', \'full_pipeline_seconds\']}, separators=(\',\', \':\')))\n', 'final_session.py': '"""Small independent session watchdog and single-launch benchmark suite (no Kaggle API/secrets)."""\nimport gzip\nimport hashlib\nimport json\nimport math\nimport os\nimport signal\nimport subprocess\nimport sys\nimport threading\nimport time\nimport zipfile\nfrom pathlib import Path\nfrom final_split import fixed_split\nfrom final_vote import select, validate_answers\n\nINPUT_SHA = {\n    \'tokenizer.json\': \'261f25b5d4be006a6a00e41e0eafbe522580741da627ba670f6b9805a2659b7d\',\n    \'test.json\': \'20cacb334e3fc81291433d2173d9085b17a15e1834bd84aea67b350403182503\',\n    \'heldout4k.json\': \'28a3c0c9088c5b4a17d75d216e4abc4262cc9ca62884966c43e7ec09b6f040c2\',\n    \'heldout4k-reference-execution.json\': \'3855680e9a37292aa2e351b8e8e4f7a5cbbe76444096d136470e4a9be8797111\'}\n\n\ndef write(path, value):\n    temp = Path(path).with_suffix(\'.tmp\')\n    temp.write_text(json.dumps(value, ensure_ascii=False, indent=2)+\'\\n\')\n    temp.replace(path)\n\n\nclass Watchdog:\n    def __init__(self, output, start):\n        self.output, self.start = Path(output), start\n        self.children = set()\n        self.done = threading.Event()\n        self.triggered = False\n\n    def kill(self):\n        for pid in list(self.children):\n            try:\n                os.killpg(pid, signal.SIGKILL)\n            except ProcessLookupError:\n                pass\n\n    def arm(self):\n        def watch():\n            while not self.done.wait(.1):\n                elapsed = time.monotonic()-self.start\n                if elapsed >= 840. and not self.triggered:\n                    self.triggered = True\n                    self.kill()\n                    write(self.output/\'watchdog.json\', {\'triggered\': True, \'elapsed_seconds\': elapsed})\n                if elapsed >= 870.:\n                    self.kill()\n                    os._exit(0)\n        threading.Thread(target=watch, daemon=True, name=\'final-session-watchdog\').start()\n        return self\n\n    def finish(self):\n        self.kill()\n        self.done.set()\n\n\ndef prepare_inputs(root, target, benchmark=True):\n    """Extract only 4 small JSONs from starter ZIP; never unpack initial/control/train weights."""\n    root, target = Path(root), Path(target)\n    target.mkdir(parents=True, exist_ok=True)\n    for name, expected in INPUT_SHA.items():\n        if not benchmark and name not in (\'tokenizer.json\', \'test.json\'):\n            continue\n        direct = list(root.rglob(name))\n        if len(direct) == 1:\n            path = direct[0]\n        elif not direct:\n            archives = list(root.rglob(\'26b-training-inputs-d0-initial-and-evaluation.zip\'))\n            assert len(archives) == 1, \'Expected direct starter files or one known starter ZIP\'\n            with zipfile.ZipFile(archives[0]) as archive:\n                members = [m for m in archive.namelist() if Path(m).name == name]\n                assert len(members) == 1\n                path = target/name\n                with path.open(\'wb\') as stream:\n                    stream.write(archive.read(members[0]))\n        else:\n            raise ValueError(\'Ambiguous input \'+name)\n        assert hashlib.sha256(path.read_bytes()).hexdigest() == expected, name\n        yield name, str(path)\n\n\ndef unique(root, name):\n    paths = list(Path(root).rglob(name))\n    assert len(paths) == 1, \'Missing or ambiguous \'+name\n    return str(paths[0])\n\n\ndef stored_tune(live, heldout):\n    """Check attached L2 first4 baseline independently, including question identities."""\n    ids = fixed_split()[\'tune2048\']\n    models = {}\n    for name in [\'C-main\', \'C-long2x\']:\n        pattern = f\'model-{name}__points__l2-28800__evaluation__k8-tune2048__batch-*.json*\'\n        files = sorted(Path(live).rglob(pattern))\n        assert len(files) == 64, \'Expected all L2 K8 journals: \'+name\n        rows = []\n        for path in files:\n            data = gzip.decompress(path.read_bytes()) if path.name.endswith(\'.gz\') else path.read_bytes()\n            rows.extend(json.loads(data))\n        index = {r[\'id\']: r for r in rows}\n        assert len(rows) == len(index) == 2048 and sorted(index) == ids\n        for i in ids:\n            assert index[i][\'question_sha256\'] == hashlib.sha256(heldout[i][\'question\'].encode()).hexdigest()\n        models[name] = index\n    matches = 0\n    for i in ids:\n        main, long = models[\'C-main\'][i], models[\'C-long2x\'][i]\n        assert main[\'reference\'] == long[\'reference\']\n        win = select(main[\'greedy\'], main[\'samples\'][\'0.7\'][:4], long[\'samples\'][\'0.7\'][:4])\n        ref = main[\'reference\']\n        matches += ref[\'status\'] == win[\'execution\'][\'status\'] == \'numeric\' and math.isclose(\n            ref[\'value\'], win[\'execution\'][\'value\'], rel_tol=1e-6, abs_tol=1e-6)\n    assert matches == 1492\n    return models, {\'matches\': matches, \'n\': 2048, \'accuracy_pp\': 100*matches/2048}\n\n\ndef diagnose(records, journals, ids):\n    result = {}\n    for name, target in [(\'C-main\', \'main\'), (\'C-long2x\', \'long\')]:\n        code_diff = sum(records[j][target][s][\'code\'] != journals[name][i][\'samples\'][\'0.7\'][s][\'code\']\n                        for j, i in enumerate(ids) for s in range(min(4, len(records[j][target]))))\n        complete = sum(len(r[target]) == 4 for r in records)\n        result[name] = {\'complete_rows\': complete, \'first4_code_mismatches\': code_diff}\n    result[\'greedy_code_mismatches\'] = sum(r[\'greedy\'][\'code\'] != journals[\'C-main\'][i][\'greedy\'][\'code\']\n                                         for r, i in zip(records, ids))\n    return result\n\n\ndef suite(inputs_root, live_root, output, module_dir, watchdog, benchmark=True):\n    output = Path(output)\n    report = {\'status\': \'running\', \'test\': [], \'GPU_runs\': 0, \'training\': False,\n              \'budget_seconds\': 900, \'internet\': False, \'single_gpu\': True}\n    write(output/\'summary.json\', report)\n    try:\n        inputs = dict(prepare_inputs(inputs_root, output/\'inputs\', benchmark=benchmark))\n        models = {name: unique(live_root, f\'model-{name}__weights__l2-28800-fp16.pt\')\n                  for name in [\'C-main\', \'C-long2x\']}\n        heldout = json.loads(Path(inputs[\'heldout4k.json\']).read_text()) if benchmark else None\n        ids = fixed_split()[\'tune2048\']\n        rows_path, ref_path = output/\'tune2048.json\', output/\'tune2048-reference.json\'\n        if benchmark:\n            references = json.loads(Path(inputs[\'heldout4k-reference-execution.json\']).read_text())\n            refs = references[\'rows\'] if isinstance(references, dict) else references\n            assert len(heldout) == len(refs) == 4096\n            write(rows_path, [{\'question\': heldout[i][\'question\']} for i in ids])\n            write(ref_path, [refs[i] for i in ids])\n            journals, report[\'stored_tune\'] = stored_tune(live_root, heldout)\n            for i in ids:\n                assert journals[\'C-main\'][i][\'reference\'] == refs[i]\n        base = {\'models\': models, \'tokenizer\': inputs[\'tokenizer.json\'],\n                \'tokenizer_sha\': INPUT_SHA[\'tokenizer.json\'], \'workers\': os.cpu_count() or 1,\n                \'budget_seconds\': 180., \'sample_batch\': 256, \'greedy_batch\': 1024}\n        def launch(name, rows, expected, **extra):\n            assert not watchdog.triggered and time.monotonic()-watchdog.start < 660.\n            destination = output/name\n            destination.mkdir(exist_ok=True)\n            config = dict(base, rows=str(rows), output=str(destination), expected_rows=expected, **extra)\n            config_path = destination/\'config.json\'; write(config_path, config)\n            env = dict(os.environ, CUDA_VISIBLE_DEVICES=\'0\')\n            log_path = destination/\'supervisor.log\'\n            with log_path.open(\'w\') as log:\n                # Timestamp before creation includes Python interpreter startup and all imports.\n                env[\'FINAL_PROCESS_START\'] = str(time.monotonic())\n                child = subprocess.Popen([sys.executable, str(Path(module_dir)/\'final_runner.py\'), str(config_path)],\n                                         start_new_session=True, env=env, stdout=log, stderr=log)\n                watchdog.children.add(child.pid)\n                try:\n                    until = min(watchdog.start+840., time.monotonic()+config[\'budget_seconds\']+5.)\n                    next_heartbeat = time.monotonic()+30.\n                    while child.poll() is None and time.monotonic() < until and not watchdog.triggered:\n                        if time.monotonic() >= next_heartbeat:\n                            print(name, \'running; session_s\', round(time.monotonic()-watchdog.start), flush=True)\n                            next_heartbeat += 30.\n                        time.sleep(.1)\n                    if child.poll() is None:\n                        os.killpg(child.pid, signal.SIGKILL)\n                    child.wait(timeout=1.)\n                    assert child.returncode == 0, name+\' failed; inspect small worker/supervisor logs\'\n                finally:\n                    watchdog.children.discard(child.pid)\n            validate_answers(destination/\'answers.jsonl\', expected)\n            report[\'GPU_runs\'] += 1\n            return json.loads((destination/\'summary.json\').read_text())\n        for number in range(1, 3 if benchmark else 2):\n            row = launch(f\'test-{number}\', inputs[\'test.json\'], 1919)\n            report[\'test\'].append(row)\n            if number == 1:\n                # The published answer is always the FIRST cold test run, never picked by heldout score.\n                (output/\'answers.jsonl\').write_bytes((output/\'test-1/answers.jsonl\').read_bytes())\n            write(output/\'summary.json\', report)\n            print(\'test\', number, row[\'status\'], round(row[\'full_pipeline_seconds\'], 3), flush=True)\n        if not benchmark:\n            report[\'status\'] = report[\'test\'][0][\'status\']\n            return report\n        report[\'tune\'] = launch(\'tune\', rows_path, 2048, reference=str(ref_path))\n        quality_path = output/\'tune/quality.json\'\n        report[\'quality\'] = json.loads(quality_path.read_text()) if quality_path.exists() else {\'passed\': False, \'reason\': \'supervisor fallback\'}\n        if (output/\'tune/candidate-journal.json.gz\').exists():\n            records = json.loads(gzip.decompress((output/\'tune/candidate-journal.json.gz\').read_bytes()))\n            report[\'K4_diagnostic\'] = diagnose(records, journals, ids)\n        write(output/\'summary.json\', report)\n        digests = [hashlib.sha256((output/f\'test-{i}/answers.jsonl\').read_bytes()).hexdigest()\n                   for i in range(1, 3)]\n        report[\'answers_sha256\'] = digests\n        report[\'deterministic_answers\'] = len(set(digests)) == 1\n        report[\'gates\'] = {\'test_all_under_180\': all(r[\'under_180_seconds\'] for r in report[\'test\']),\n                           \'test_all_under_120\': all(r[\'target_120_seconds\'] for r in report[\'test\']),\n                           \'no_test_fallback\': all(r[\'status\'] == \'complete\' and\n                               r[\'deadline_guard\'][\'fallback_count\'] == 0 for r in report[\'test\']),\n                           \'deterministic_answers\': report[\'deterministic_answers\'],\n                           \'quality_within_1pp\': report[\'quality\'][\'passed\']}\n        report[\'status\'] = \'passed\' if all(report[\'gates\'].values()) else \'review_required\'\n    except BaseException as error:\n        report.update(status=\'failed\', error_type=type(error).__name__, error=str(error))\n        raise\n    finally:\n        report[\'session_seconds\'] = time.monotonic()-watchdog.start\n        write(output/\'summary.json\', report)\n        watchdog.finish()\n    return report\n'}
for name,source in SOURCES.items():(MODULES/name).write_text(source)
sys.path.insert(0,str(MODULES))
from final_session import Watchdog
WATCHDOG=Watchdog(OUTPUT,SESSION_START).arm()


В Kaggle пути находятся автоматически. Для Colab смонтируйте те же входы и измените две переменные ниже; suite оставляет отдельные файлы каждого прогона и summary. Подготовка ZIP и проверка входных SHA входят в бюджет сессии; каждый холодный test повторно читает test, проверяет и загружает оба FP16 файла/tokenizer внутри своего таймера.

In [ ]:
BENCHMARK=True  # Ревью: 2 test + tune. Для сдачи после GPU gate: False (один test).
def find_input(slug,root=Path('/kaggle/input')):
    # Kaggle mounts either /kaggle/input/<slug> or /kaggle/input/datasets/<owner>/<slug>.
    if (root/slug).is_dir():return root/slug
    hits=[p for p in root.rglob(slug) if p.is_dir()] if root.is_dir() else []
    assert len(hits)==1,f'Cannot resolve input {slug}: {[str(p) for p in hits]}'
    return hits[0]
INPUTS_ROOT=find_input('nlp-hw02-26b-inputs')
LIVE_ROOT=find_input('nlp-hw02-l2-live')
assert INPUTS_ROOT.is_dir() and LIVE_ROOT.is_dir(),'Mount both private datasets or set their Colab paths'
from final_session import suite
REPORT=suite(INPUTS_ROOT,LIVE_ROOT,OUTPUT,MODULES,WATCHDOG,benchmark=BENCHMARK)
print(json.dumps({'status':REPORT['status'],'test_seconds':[r['full_pipeline_seconds'] for r in REPORT['test']],
                 'quality':REPORT.get('quality'),'gates':REPORT.get('gates'),'session_seconds':REPORT['session_seconds']},indent=2))


Проверяйте `final-inference/summary.json`, `test-1..2/timing.json`, `tune/quality.json`. Guard голосует по уже исполненным samples; без samples остаётся greedy. Счётчики partial vote/greedy-only записываются; такой прогон не подтверждает полный mixed gate. Итоговый `final-inference/answers.jsonl` можно отправлять боту после независимого ревью и успешного GPU gate. Для будущей сдачи можно оставить один test-прогон после проверки двух таймеров; текущий notebook — измерительный вариант с пояснениями.